# Customer Lifetime Value (CLV) Modeling

In this notebook we estimate **Customer Lifetime Value (CLV)** using probabilistic customer behavior models.

Customer Lifetime Value represents the **expected future revenue generated by a customer** over a defined time horizon.

While the previous notebook focused on **churn prediction**, CLV modeling helps quantify **how valuable each customer is likely to be in the future**.

We use the widely adopted **BG/NBD + Gamma-Gamma modeling framework**, which is designed for non-contractual businesses such as e-commerce where customers are free to purchase at irregular intervals.

The notebook includes:

- preparing transaction data for CLV modeling
- calculating customer behavioral summaries
- fitting a **BG/NBD model** to estimate purchase frequency
- fitting a **Gamma-Gamma model** to estimate average monetary value
- estimating **future customer lifetime value**
- segmenting customers by predicted value
- saving CLV outputs for business analysis

This notebook builds on the previous steps in the portfolio:

01 — Data Understanding  
02 — Customer Base Table  
03 — RFM Segmentation  
04 — Churn Prediction  
05 — **Customer Lifetime Value (CLV) Modeling**

We import the libraries required for data processing, visualization, and CLV modeling.

The **Lifetimes** library provides implementations of the BG/NBD and Gamma-Gamma models widely used for customer lifetime value estimation.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

pd.set_option("display.max_columns", None)

To estimate CLV we require historical transaction data at the order level.

We load the following Olist datasets:

- orders dataset containing purchase timestamps
- customers dataset mapping orders to unique customers
- payments dataset containing transaction value

We will merge these datasets to create a clean transaction-level table suitable for CLV modeling.

In [ ]:
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")

orders.shape, customers.shape, payments.shape

We convert purchase timestamps to datetime format and inspect the time span covered by the dataset.

CLV models rely on knowing the **customer observation period**, which is the time range over which we observe customer purchases.

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

We now create a clean transaction-level table.

Steps include:

1. keeping only delivered orders
2. mapping customer identifiers
3. aggregating payment values at the order level

This results in a dataset with one row per completed purchase.

In [ ]:
order_payments = (
    payments.groupby("order_id", as_index=False)["payment_value"]
    .sum()
)

transactions = (
    orders.loc[orders["order_status"] == "delivered", [
        "order_id",
        "customer_id",
        "order_purchase_timestamp"
    ]]
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .merge(order_payments, on="order_id")
)

transactions.head()

CLV models use transaction **dates rather than timestamps**.

We therefore normalize purchase timestamps to dates.

In [ ]:
transactions["order_date"] = transactions[
    "order_purchase_timestamp"
].dt.date

transactions.head()

The Lifetimes package requires a **customer summary table** containing:

frequency — number of repeat purchases  
recency — time between first and last purchase  
T — customer observation period  
monetary_value — average order value

We compute these metrics from the transaction dataset.

In [ ]:
clv_summary = summary_data_from_transaction_data(
    transactions,
    customer_id_col="customer_unique_id",
    datetime_col="order_date",
    monetary_value_col="payment_value",
    observation_period_end=transactions["order_date"].max()
)

clv_summary.head()

Before fitting CLV models we inspect the behavioral summary metrics to understand the customer purchasing patterns.

In [ ]:
clv_summary.describe()

Purchase frequency distribution provides insight into customer repeat purchasing behavior.

As observed in earlier notebooks, many customers place only a single order, which is typical in many e-commerce businesses.

In [ ]:
clv_summary["frequency"].hist(bins=30)
plt.title("Distribution of Repeat Purchase Frequency")
plt.xlabel("Repeat Purchases")
plt.ylabel("Number of Customers")
plt.show()

The **BG/NBD (Beta Geometric / Negative Binomial Distribution)** model estimates:

- how frequently customers purchase
- the probability that a customer is still active

This allows us to predict **expected future transactions** for each customer.

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.001)

bgf.fit(
    clv_summary["frequency"],
    clv_summary["recency"],
    clv_summary["T"]
)

Using the fitted BG/NBD model we estimate the **expected number of future purchases** for each customer over the next 6 months.

In [ ]:
clv_summary["predicted_purchases_6m"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    180,
    clv_summary["frequency"],
    clv_summary["recency"],
    clv_summary["T"]
)

clv_summary.head()

The Gamma-Gamma model requires customers with at least one repeat purchase.

Customers with only a single purchase cannot provide reliable estimates of average spending behavior.

In [ ]:
returning_customers = clv_summary[
    clv_summary["frequency"] > 0
].copy()

returning_customers.shape

The **Gamma-Gamma model** estimates the expected average monetary value of future transactions.

It assumes that spending behavior is independent of purchase frequency.

In [ ]:
ggf = GammaGammaFitter(penalizer_coef=0.001)

ggf.fit(
    returning_customers["frequency"],
    returning_customers["monetary_value"]
)

We now combine the BG/NBD and Gamma-Gamma models to estimate **Customer Lifetime Value**.

We calculate CLV over a **6 month horizon**, discounted to reflect the time value of money.

In [ ]:
clv_summary["clv_6m"] = ggf.customer_lifetime_value(
    bgf,
    clv_summary["frequency"],
    clv_summary["recency"],
    clv_summary["T"],
    clv_summary["monetary_value"],
    time=6,
    freq="D",
    discount_rate=0.01
)

clv_summary.head()

Customer Lifetime Value distributions are usually highly skewed, with a small group of customers generating a large proportion of future revenue.

In [ ]:
clv_summary["clv_6m"].hist(bins=50)
plt.title("Distribution of Predicted 6-Month CLV")
plt.xlabel("CLV")
plt.ylabel("Customers")
plt.show()

To make CLV easier to use in business decisions, we segment customers into value tiers.

These segments allow businesses to differentiate between:

- high value customers
- medium value customers
- low value customers

In [ ]:
clv_summary["clv_segment"] = pd.qcut(
    clv_summary["clv_6m"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

clv_summary["clv_segment"].value_counts()

We inspect the highest predicted CLV customers, which represent the most valuable customers for retention and marketing strategies.

In [ ]:
clv_summary.sort_values(
    "clv_6m",
    ascending=False
).head(10)

We save the CLV results so they can be used in the final notebook that focuses on business recommendations and retention strategy.

In [ ]:
clv_summary.to_csv(
    OUTPUT_DIR / "customer_clv_scores.csv",
    index=True
)

To better prioritize customers, we combine predicted **churn probability** from the churn model with **customer lifetime value (CLV)** estimated in this notebook.

This allows us to create a **customer value matrix**, which helps identify customers who are both high value and at risk of churn.

In [ ]:
churn_scores = pd.read_csv(
    OUTPUT_DIR / "customer_churn_scores.csv"
)

churn_scores.head()

We merge the churn predictions with CLV estimates to create a unified customer analytics table.

In [ ]:
customer_value_df = (
    clv_summary.reset_index()
    .merge(
        churn_scores[["customer_unique_id", "churn_probability"]],
        on="customer_unique_id",
        how="left"
    )
)

customer_value_df.head()

We now categorize customers into segments based on:

- predicted churn risk
- predicted lifetime value

This creates a simple but powerful **customer value matrix** used in retention strategy.

In [ ]:
customer_value_df["clv_tier"] = pd.qcut(
    customer_value_df["clv_6m"],
    2,
    labels=["Low CLV", "High CLV"]
)

customer_value_df["churn_risk"] = np.where(
    customer_value_df["churn_probability"] > 0.6,
    "High Risk",
    "Low Risk"
)

customer_value_df["value_segment"] = (
    customer_value_df["clv_tier"].astype(str)
    + " | "
    + customer_value_df["churn_risk"]
)

customer_value_df["value_segment"].value_counts()

## Conclusion

In this notebook we estimated Customer Lifetime Value using probabilistic behavioral models.

Key outcomes include:

- predicting future customer purchase frequency
- estimating expected transaction value
- calculating customer lifetime value
- segmenting customers by expected future value

These CLV estimates provide an important complement to the churn predictions from the previous notebook.

Together they enable businesses to prioritize customer retention efforts by focusing on customers who are both:

- **at risk of churn**
- **high expected lifetime value**

The next notebook will combine churn risk, RFM segmentation, and CLV predictions to produce actionable **business recommendations for customer retention and growth**.

## Linking CLV and Churn for Business Strategy

By combining CLV predictions with churn risk estimates from the previous notebook, we can identify customers who are both:

- high expected lifetime value
- at risk of churn

These customers represent the highest priority targets for retention campaigns.

In the next notebook we will combine RFM segments, churn predictions, and CLV estimates to build a customer prioritization framework and simulate retention strategies.